# Signisa — Phase 1 diagnostic (CPU, internet ON)

Both losses hit the same ~73% TAR@FAR5 wall -> suspect a shared cause in the data or a
val signer, not the loss. Attach TWO inputs: the `kaggle_prep` output (tensors) and the
ArcFace `kaggle_train` output (model.pt). Paths are discovered by glob — no hardcoded mounts.

Reports: TAR/EER by val participant (+ mirrored flags), TAR by sign ascending, the 100
worst genuine trials with the centroid that beat them, and genuine-score spread per signer.
Writes `diagnosis_report.md` + `worst_genuine.csv`.

In [ ]:
from pathlib import Path

inputs = Path("/kaggle/input")
index_csvs = sorted(inputs.glob("*/tensors/index.csv")) or sorted(inputs.rglob("index.csv"))
models = sorted(inputs.rglob("model.pt"))
assert len(index_csvs) == 1, f"attach exactly ONE prep output; found {index_csvs}"
assert len(models) == 1, f"attach exactly ONE train output; found {models}"
tensors_dir = index_csvs[0].parent
model_pt = models[0]
print("tensors:", tensors_dir)
print("model:  ", model_pt)

In [ ]:
!git clone -q https://github.com/dayan-battulga/signisa.git /kaggle/working/signisa-repo
%pip install -q /kaggle/working/signisa-repo

In [ ]:
import torch

from signisa.config import Config
from signisa.eval import run_evaluation
from signisa.models import SignModel

state = torch.load(model_pt, map_location="cpu")
loss = "ce" if "head.bias" in state else "arcface"
cfg = Config(loss=loss)
model = SignModel(cfg)
model.load_state_dict(state)
print("loaded", loss, "model")

REPO = "/kaggle/working/signisa-repo"
metrics = run_evaluation(
    model, tensors_dir, f"{REPO}/data/meta/curriculum_db.json",
    f"{REPO}/data/meta/training_labels.json", cfg, "/kaggle/working", device="cpu")
print(f"recomputed: TAR@FAR5 {metrics['tar_at_far']:.1%}, "
      f"top-1 {metrics['top1_closed_set']:.1%}")

In [ ]:
import json

import pandas as pd

trials = metrics["trials"]
thr = metrics["global_far_threshold"]
label_of = {c["id"]: c["label"] for c in json.load(
    open(f"{REPO}/data/meta/training_labels.json"))["classes"]}

# (a + d) per-participant: TAR, mean EER, genuine spread, mirrored flag
pp = pd.DataFrame(metrics["per_participant"]).T
pp.index.name = "participant"
print(pp.round(3).to_string(), "\n")

# (b) per-sign TAR at the global threshold, ascending
g = trials[trials.genuine].copy()
g["hit"] = g.score >= thr
by_sign = (g.groupby("target")
             .agg(n=("score", "size"), tar=("hit", "mean"), median_score=("score", "median"))
             .rename(index=label_of).sort_values("tar"))
print("worst signs by TAR:\n", by_sign.head(15).round(3).to_string(), "\n")

# (c) 100 worst genuine trials with margin and the centroid that beat them
imp = trials[~trials.genuine]
best = imp.loc[imp.groupby("sequence_id").score.idxmax(),
               ["sequence_id", "target", "score"]]
best.columns = ["sequence_id", "beaten_by_id", "best_impostor"]
worst = g.merge(best, on="sequence_id", how="left")
worst["margin"] = worst.score - worst.best_impostor
worst["sign"] = worst.target.map(label_of)
worst["beaten_by"] = worst.beaten_by_id.map(label_of)
worst = worst.sort_values("score").head(100)
cols = ["sequence_id", "participant", "sign", "score", "margin", "beaten_by", "best_impostor"]
worst[cols].round(4).to_csv("/kaggle/working/worst_genuine.csv", index=False)

def md_table(df):
    header = "| " + " | ".join(str(c) for c in [df.index.name or ""] + list(df.columns)) + " |"
    sep = "|" + "---|" * (len(df.columns) + 1)
    rows = ["| " + " | ".join(str(v) for v in [i] + list(r)) + " |"
            for i, r in zip(df.index, df.round(3).values)]
    return "\n".join([header, sep] + rows)

report = "\n\n".join([
    "# Phase 1 diagnostic",
    f"Global (recomputed): TAR@FAR5 {metrics['tar_at_far']:.1%}, "
    f"top-1 {metrics['top1_closed_set']:.1%}, threshold {thr:.3f}, loss={cfg.loss}.",
    "## Per-participant (bad-signer / wrong-mirroring check)", md_table(pp),
    "## Bottom 20 signs by TAR", md_table(by_sign.head(20)),
    f"## Worst genuine trials (top 20 of {len(worst)}; full list in worst_genuine.csv)",
    md_table(worst[cols].set_index("sequence_id").head(20)),
])
Path("/kaggle/working/diagnosis_report.md").write_text(report + "\n")

spread = pp.tar_at_far.max() - pp.tar_at_far.min()
misses = by_sign[by_sign.tar <= by_sign.tar.quantile(0.2)].n.sum() / by_sign.n.sum()
print(f"summary: participant TAR spread {spread:.1%} "
      f"(concentrated signer problem if large); "
      f"bottom-20%-of-signs hold {misses:.1%} of genuine trials; "
      f"{(worst.margin < 0).sum()}/{len(worst)} worst genuines actually beaten by another centroid")